Modeling Features: DSPL
=======================

A double source-plane lens (DSPL) is a strong lens system where there are two source galaxies at different
redshifts behind the lens galaxy. They appear as two distinct Einstein rings in the image-plane, and can constrain
Cosmological parameters in a way single Einstein ring lenses cannot.

To analyse these systems correctly the mass of the lens galaxy and the first source galaxy must be modeled
simultaneously, and the emission of both source galaxies must be modeled simultaneously.

This script illustrates the PyAutoLens API for modeling a DSPL.

__Practical Use: Read This First__

This script is a tutorial. It produces a working fit by "cheating" — every prior is initialised at the true
simulator value, narrowed by a small Gaussian. On real data this is impossible, and a single Nautilus search on
a 16-parameter DSPL model would almost certainly converge to a local maximum.

The script you will actually use to fit a DSPL on real data is
`autolens_workspace/scripts/imaging/features/advanced/double_source_plane_lens/chaining.py`, which runs two chained
non-linear searches: the first initialises the lens mass and `source_0` using a smaller mask that excludes
`source_1`, the second introduces `source_1` and frees `source_0`'s mass. This is also significantly more
computationally efficient than the single-search approach below.

For production-quality modeling, see `slam.py` in the same directory.

Read this script to understand the model composition API, then jump to `chaining.py`.

__Contents__

- **Model:** Compose the lens model fitted to the data.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Over Sampling:** Set up the adaptive over-sampling grid for accurate light profile evaluation.
- **Model Cookbook:** A full description of model composition is provided by the model cookbook.
- **Cheating:** Initializing a DSPL model is difficult, due to the complexity of parameter.
- **Cosmology:** DSPLs allow cosmological parameters to be constrained — `Om0` is fixed at
  Planck18 here, with a commented-out snippet showing how to make it free.
- **Search:** Configure the non-linear search used to fit the model.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **VRAM:** The `modeling` example explains how VRAM is used during GPU-based fitting and how to print the.
- **Run Time:** Profiling the expected run time of the model-fit.
- **Result:** Overview of the results of the model-fit.
- **Wrap Up:** Summary of the script and next steps.

__Model__

This script fits an `Imaging` dataset of a 'galaxy-scale' strong lens which is a DSPL where:

 - The lens galaxy's light is omitted (and is not present in the simulated data).
 - The first lens galaxy's total mass distribution is an `Isothermal`.
 - The second lens galaxy / first source galaxy's light is a linear `ExponentialSph` and its mass a `IsothermalSph`.
 - The second source galaxy's light is a linear `ExponentialSph`.

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/start_here.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

# from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset__

Load and plot the strong lens dataset `double_source_plane_lens` via .fits files.

This dataset has a double Einstien ring, due to the two source galaxies at different redshifts behind the lens galaxy.

In [ ]:
dataset_name = "double_source_plane_lens"
dataset_path = Path("dataset") / "imaging" / dataset_name

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "scripts/imaging/features/advanced/double_source_plane_lens/simulator.py",
        ],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

Visualization of this dataset shows two distinct Einstein rings, which are the two source galaxies.

In [ ]:
aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

Define a 3.0" circular mask, which includes the emission of both of the lensed source galaxies.

In [ ]:
mask_radius = 3.0

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

aplt.subplot_imaging_dataset(dataset=dataset)

__Over Sampling__

Apply adaptive over sampling to ensure the lens galaxy light calculation is accurate, you can read up on over-sampling 
in more detail via the `autolens_workspace/*/guides/advanced/over_sampling.ipynb` notebook.

In [ ]:
over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=dataset.grid,
    sub_size_list=[4, 2, 2],
    radial_list=[0.3, 0.6],
    centre_list=[(0.0, 0.0)],
)

dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

aplt.subplot_imaging_dataset(dataset=dataset)

__Model__

We compose a lens model where:

 - The first lens galaxy's total mass distribution is an `Isothermal` [5 parameters].

 - The second lens / first source galaxy's light are MGE models [8 parameters].

 - The second source galaxy's light is a linear `ExponentialSph` [3 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=16.

Note that the galaxies are assigned redshifts of 0.5, 1.0 and 2.0. This ensures the multi-plane ray-tracing necessary
for the DSPL is performed correctly.

The `centre` values input into `mge_model_from` are explained below.

__Model Cookbook__

A full description of model composition is provided by the model cookbook: 

https://pyautolens.readthedocs.io/en/latest/general/model_cookbook.html

In [ ]:
# Lens:

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    centre_prior_is_uniform=True,
    centre=(0.0, 0.0),
    sigma_min=dataset.pixel_scales[0] / 10.0,
)
mass = af.Model(al.mp.Isothermal)

lens = af.Model(al.Galaxy, redshift=0.5, bulge=bulge, mass=mass)

# Source 0:

bulge = af.Model(al.lp_linear.ExponentialCoreSph)
mass = af.Model(al.mp.IsothermalSph)

source_0 = af.Model(
    al.Galaxy, redshift=1.0, bulge=bulge, mass=mass, centre=(0.15, 0.15)
)

# Source 1:

bulge = af.Model(al.lp_linear.ExponentialCoreSph)

source_1 = af.Model(al.Galaxy, redshift=2.0, bulge=bulge, centre=(0.0, 0.0))

__Cheating__

Initializing a DSPL model is difficult, due to the complexity of parameter space. It is common to 
infer local maxima, which this script does if default broad priors on every model parameter are assumed.

To infer the correct model, we "cheat" and overwrite all of the priors of the model parameters to start centred on 
their true values. This is why the true `centre` values were input into the `mge_model_from` functions above.

For real data, we obviously do not know the true parameters and therefore cannot cheat in this way. Readers should
checkout the **PyAutoLens**'s advanced feature `chaining`, which chains together multiple non-linear searches. 

This feature is described in HowToLens chapter 3 and specific examples for a DSPL are given in
the script `imaging/features/advanced/double_source_plane_lens/chaining.py`.

In [ ]:
lens.mass.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.1)
lens.mass.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.1)

source_0.mass.centre_0 = af.GaussianPrior(mean=-0.15, sigma=0.2)
source_0.mass.centre_1 = af.GaussianPrior(mean=-0.15, sigma=0.2)
source_0.mass.einstein_radius = af.GaussianPrior(mean=0.4, sigma=0.1)

__Cosmology__

DSPLs allow cosmological parameters to be constrained, because they provide information on the
different angular diameter distances between the lens, `source_0` and `source_1`. The deflection scaling factor
between the two source-planes (sometimes written `beta_01`) depends on those distances and therefore on the
cosmology.

For this tutorial, we use a fixed `Planck18` cosmology and treat no cosmological parameters as free. This keeps
the model dimensionality and the parameter space tractable for the single-search "cheating" workflow below; a
realistic cosmological constraint requires both the chained-search workflow in `chaining.py` and significantly
more data than this one simulated system.

To make `Om0` (Omega_m) a free parameter in your own fit, uncomment the three lines below. They construct a
`FlatLambdaCDM` cosmology as a free model, override the prior on `Om0`, and include the cosmology in the overall
`Collection`. The remaining cosmological parameters (`H0`, `Tcmb0`, etc.) stay fixed at their Planck18 values.

In [ ]:
# cosmology = af.Model(al.cosmo.FlatLambdaCDM)
# cosmology.Om0 = af.GaussianPrior(mean=0.3, sigma=0.1)

# Overall Lens Model:

model = af.Collection(
    galaxies=af.Collection(lens=lens, source_0=source_0, source_1=source_1),
    # cosmology=cosmology,
)

The `info` attribute shows the model in a readable format.

This confirms the model is composed of three galaxies, two of which are lensed source galaxies, and a fixed
Planck18 cosmology.

In [ ]:
print(model.info)

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("imaging") / "features",
    name="double_source_plane_lens",
    unique_tag=dataset_name,
    n_live=150,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    iterations_per_quick_update=2000,
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Analysis__

Create the `AnalysisImaging` object defining how the via Nautilus the model is fitted to the data.

In [ ]:
analysis = al.AnalysisImaging(dataset=dataset, use_jax=True)

__VRAM__

The `modeling` example explains how VRAM is used during GPU-based fitting and how to print the estimated VRAM 
required by a model.

Double source plane lenses can use a lot of VRAM, because the multi-plane ray-tracing and creation of multiple
images for different source planes can require all the additional data to be stored in VRAM. This will
at least double the VRAM requirements compared to a single lens plane model, but often more than this.

Given VRAM use is an important consideration, we print out the estimated VRAM required for this
model-fit and advise you do this for your own double source plane lens model-fits.

The method below prints the VRAM usage estimate for the analysis and model with the specified batch size,
it takes about 20-30 seconds to run so you may want to comment it out once you are familiar with your GPU's VRAM limits.

In [ ]:
analysis.print_vram_use(model=model, batch_size=search.batch_size)

__Run Time__

The likelihood evaluation time for analysing a DSPL is quite a lot longer than single lens plane
lenses. This is because multi-plane ray-tracing calculations are computationally expensive. 

However, the real hit on run-time is the large number of free parameters in the model, which is often  10+ parameters
more than a single lens plane model. This means that the non-linear search takes longer to converge on a solution.
In this example, we cheated by initializing the priors on the model close to the correct solution. 

Combining pixelized source analyses with DSPLs is very computationally expensive, because the
linear algebra calculations become significantly more expensive. This is not shown in this script, but is worth
baring in mind.

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

In [ ]:
result = search.fit(model=model, analysis=analysis)

__Result__

The search returns a result object, which whose `info` attribute shows the result in a readable format (if this does not display clearly on your screen refer to
`start_here.ipynb` for a description of how to fix this):

In [ ]:
print(result.info)

We plot the maximum likelihood fit, tracer images and posteriors inferred via Nautilus.

These plots show that the lens and both sources of the DSPL were fitted successfully.

In [ ]:
print(result.max_log_likelihood_instance)

aplt.subplot_tracer(tracer=result.max_log_likelihood_tracer, grid=result.grids.lp)

aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

aplt.corner_anesthetic(samples=result.samples)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.

These examples show how the results API can be extended to investigate DSPL results.

__Wrap Up__

DSPLs can be fitted in **PyAutoLens**, however this script bypass the most difficult aspect
of fitting these systems by "cheating", and manually adjusting the priors to be near their true values.

Modeling real observations of DSPLs is one of the hardest lens modeling tasks, and requires an high
degree of lens modeling expertise to make a success.

If you have not already, I recommend you familiarize yourself with and use all of the following **PyAutoLens features
to model a real DSPL:

 - Basis based light profiles (e.g. `features/advanced/shapelets/modeling.ipynb` / `features/multi_gaussian_expansion/modeling.ipynb`): these allow one to fit
   complex lens and source morphologies whilst keeping the dimensionality of the problem low.

 - Search chaining (e.g. `guides/modeling/chaining` and HowToLens chapter 3): by breaking the model-fit into a series
   of Nautilus searches models of gradually increasing complexity can be fitted.

 - pixelizations (e.g. `features/pixelization/modeling.ipynb` and HowToLens chapter 4): to infer the cosmological parameters reliably
   the source must be reconstructed on an adaptive mesh to capture a irregular morphological features.